# Heavy-Tailed ABMs: Cont-Bouchaud & OFC

This notebook reproduces power-law experiments from two agent-based models:

1. **Cont-Bouchaud (CB)**: herd behaviour and fat tails in financial markets.
2. **Olami-Feder-Christensen (OFC)**: self-organised criticality in seismic faults.

Sections:
1. Setup & imports
2. CB simulation demo
3. CB phase diagram
4. CB calibration (MLE + ABC)
5. OFC simulation demo
6. OFC phase diagram
7. OFC calibration (MLE + ABC)
8. Comparison & discussion

## 1. Setup & imports

In [1]:
import os
import sys
import warnings

warnings.filterwarnings('ignore')

# Ensure the project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.dirname('.'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import matplotlib
matplotlib.use('Agg')  # headless backend; switch to 'inline' in Jupyter
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image

from models.cont_bouchaud import simulate_cb
from models.ofc import simulate_ofc
from utils.powerlaw_fit import fit_powerlaw, gutenberg_richter_b
from utils.plotting import (
    plot_ccdf, plot_return_series, plot_comparison,
    FIGURES_DIR, _apply_style, _ensure_figures_dir)

_ensure_figures_dir()
print('Setup complete. Figures will be saved to:', FIGURES_DIR)

Setup complete. Figures will be saved to: /Users/gabriel/Desktop/heavy_tails_abm/figures


## 2. Cont-Bouchaud simulation demo

Single run at `p=0.59` (near the percolation threshold), `L=64`, 10 000 steps.

In [2]:
# CB single run
CB_L, CB_P, CB_A, CB_STEPS = 64, 0.59, 0.01, 10_000
returns_cb = simulate_cb(CB_L, CB_P, CB_A, CB_STEPS, seed=42)

print(f'CB returns: mean={returns_cb.mean():.4f}, std={returns_cb.std():.4f}')
print(f'Kurtosis: {float(np.mean((returns_cb - returns_cb.mean())**4) / returns_cb.std()**4):.2f}')

CB returns: mean=-2.5629, std=513.5435
Kurtosis: 54.08


In [3]:
# Return time series
save_ts = os.path.join(FIGURES_DIR, 'cb_demo_timeseries.pdf')
plot_return_series(returns_cb, title='CB returns (p=0.59, L=64)', save_path=save_ts)
print('Saved:', save_ts)

Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/cb_demo_timeseries.pdf


In [4]:
# CCDF of absolute returns
_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))

abs_ret = np.abs(returns_cb[returns_cb != 0])
fit_res = fit_powerlaw(abs_ret)
print(f"Power-law alpha = {fit_res['alpha']:.3f} (xmin={fit_res['xmin']:.4f})")
print(f"vs lognormal: R={fit_res['R_lognormal']:.3f}, p={fit_res['p_lognormal']:.3f}")

plot_ccdf(abs_ret, label='CB |returns|', ax=ax, color='steelblue', fit_result=fit_res)
ax.set_title(f'CB CCDF (p={CB_P}, L={CB_L})')

save_ccdf = os.path.join(FIGURES_DIR, 'cb_demo_ccdf.pdf')
fig.tight_layout()
fig.savefig(save_ccdf)
plt.close(fig)
print('Saved:', save_ccdf)

Power-law alpha = 2.171 (xmin=3.0000)
vs lognormal: R=-10.371, p=0.000
Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/cb_demo_ccdf.pdf


## 3. CB Phase Diagram

Grid search over `p ∈ [0.40, 0.70]` and `L ∈ {32, 64, 128}`.

In [5]:
from experiments.cb_phase_diagram import (
    run_cb_phase_diagram, plot_cb_phase_diagram,
    P_GRID, L_VALUES as CB_L_VALUES)

cb_exponents = run_cb_phase_diagram()
plot_cb_phase_diagram(cb_exponents)

CB phase diagram: 100%|██████████| 150/150 [02:37<00:00,  1.05s/it]


Saved: /Users/gabriel/Desktop/heavy_tails_abm/figures/cb_phase_diagram.pdf


'/Users/gabriel/Desktop/heavy_tails_abm/figures/cb_phase_diagram.pdf'

## 4. CB Calibration on CAC 40 and S&P 500

### 4a. MLE calibration

In [ ]:
from data.download_data import download_stock_returns
from experiments.cb_calibration import run_mle_calibration as cb_mle

print('Downloading stock returns ...')
returns_dict = download_stock_returns(
    tickers=['^FCHI', '^GSPC'],
    start='2000-01-01',
    end='2024-01-01')

for ticker, rets in returns_dict.items():
    abs_ret = np.abs(rets.dropna().values)
    abs_ret = abs_ret[abs_ret > 0]
    res = fit_powerlaw(abs_ret)
    alpha_emp = res['alpha']
    print(f'\n{ticker}: empirical alpha = {alpha_emp:.4f}')
    cb_mle(alpha_emp, ticker)

### 4b. ABC calibration

In [ ]:
from experiments.cb_calibration import run_abc_calibration as cb_abc

for ticker, rets in returns_dict.items():
    abs_ret = np.abs(rets.dropna().values)
    abs_ret = abs_ret[abs_ret > 0]
    res = fit_powerlaw(abs_ret)
    alpha_emp = res['alpha']
    print(f'\nABC calibration for {ticker} (alpha_emp={alpha_emp:.4f}) ...')
    try:
        cb_abc(alpha_emp, ticker)
    except Exception as e:
        print(f'  Skipped: {e}')

## 5. OFC Simulation Demo

Single run with `alpha_ofc=0.20`, `L=128`, 100 000 events.

In [ ]:
OFC_L, OFC_ALPHA, OFC_EVENTS = 128, 0.20, 100_000
sizes_ofc = simulate_ofc(OFC_L, OFC_ALPHA, OFC_EVENTS, seed=42)

print(f'OFC avalanche sizes: min={sizes_ofc.min()}, max={sizes_ofc.max()}')
print(f'Mean size: {sizes_ofc.mean():.2f}')

# Avalanche time series
_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))
ax.semilogy(sizes_ofc[:2000], linewidth=0.4, color='darkorange')
ax.set_xlabel('Event')
ax.set_ylabel('Avalanche size')
ax.set_title(f'OFC avalanches (alpha={OFC_ALPHA}, L={OFC_L})')
fig.tight_layout()
save_ofc_ts = os.path.join(FIGURES_DIR, 'ofc_demo_timeseries.pdf')
fig.savefig(save_ofc_ts)
plt.close(fig)
print('Saved:', save_ofc_ts)

In [ ]:
# CCDF of avalanche sizes
_apply_style()
fig, ax = plt.subplots(figsize=(3.5, 2.8))

fit_res_ofc = fit_powerlaw(sizes_ofc.astype(float))
print(f"Power-law alpha = {fit_res_ofc['alpha']:.3f} (xmin={fit_res_ofc['xmin']:.1f})")

mags_ofc = np.log10(sizes_ofc[sizes_ofc > 0].astype(float))
b_ofc = gutenberg_richter_b(mags_ofc)
print(f'G-R b-value = {b_ofc:.4f}')

plot_ccdf(sizes_ofc.astype(float), label='OFC avalanche sizes', ax=ax,
          color='darkorange', fit_result=fit_res_ofc)
ax.set_title(f'OFC CCDF (α={OFC_ALPHA}, L={OFC_L})')

save_ofc_ccdf = os.path.join(FIGURES_DIR, 'ofc_demo_ccdf.pdf')
fig.tight_layout()
fig.savefig(save_ofc_ccdf)
plt.close(fig)
print('Saved:', save_ofc_ccdf)

## 6. OFC Phase Diagram

Grid search over `alpha_ofc ∈ [0.05, 0.24]` and `L ∈ {64, 128, 256}`.

In [ ]:
from experiments.ofc_phase_diagram import (
    run_ofc_phase_diagram, plot_ofc_phase_diagram,
    ALPHA_GRID, L_VALUES as OFC_L_VALUES)

ofc_b_values = run_ofc_phase_diagram()
plot_ofc_phase_diagram(ofc_b_values)

## 7. OFC Calibration on USGS Catalog

### 7a. MLE calibration

In [ ]:
from data.download_data import download_usgs_catalog
from experiments.ofc_calibration import run_mle_calibration as ofc_mle

print('Downloading USGS catalog ...')
catalog = download_usgs_catalog(min_magnitude=2.0, start='2000-01-01', end='2024-01-01')
magnitudes = catalog['magnitude'].dropna().values
print(f'{len(magnitudes)} events, M in [{magnitudes.min():.1f}, {magnitudes.max():.1f}]')

b_emp = gutenberg_richter_b(magnitudes)
print(f'Empirical b = {b_emp:.4f}')

alpha_star, b_sim_grid = ofc_mle(b_emp)

### 7b. ABC calibration

In [ ]:
from experiments.ofc_calibration import run_abc_calibration as ofc_abc

print(f'Running ABC calibration (b_emp={b_emp:.4f}) ...')
try:
    ofc_abc(b_emp)
except Exception as e:
    print(f'Skipped: {e}')